# Convert measured building demand data to kw and pf

## Import packages

In [ ]:
import gc
import os
import joblib
import time

from src import input_ops
from src import file_ops

## Load config file with scenarios and parameters 

In [ ]:
config_file_name = 'opendss_config1'; config_path = f"config/{config_file_name}.yaml"; config = input_ops.load_config(config_path)
    
line_rating_mode = config['line_rating_mode'] 

aggregation_level = config['aggregation_level']

TGW_years_scenarios = config['TGW_years_scenarios']

CITY_REGIONS_TO_RUN = config['CITY_REGIONS_TO_RUN']

smart_ds_years = config['smart_ds_years']

smart_ds_load_path = config['smart_ds_load_path'] + f"/{smart_ds_years[0]}" 

input_data_prediction_path = config['input_data_prediction_path']

print(f"TGW_years_scenarios: {TGW_years_scenarios} \n\ncity region: {CITY_REGIONS_TO_RUN} \n\nLoad weather data from: {input_data_prediction_path}")

## Load full measured building dictionary (with total kw, pf cooling, heating, non_cool_n_heat)

In [ ]:
## Select city and region
city = 'GSO'
region = 'rural'

smart_ds_year = config['smart_ds_years'][0]

smart_ds_load_path = config['smart_ds_load_path'] + f"/{smart_ds_year}"

input_data_region_dir = f'{smart_ds_load_path}/{city}/{region}/buildings'
measured_buildings_cool_heat_dict = joblib.load(os.path.join(input_data_region_dir, "measured_buildings_cool_heat_dict.joblib")) 

## Create a reduced form of measured building dictionary (for multiple city-region combinations) - keeping only total kw and pf 

In [ ]:
start_time = time.time()

skip_output_that_already_exists = False 

# =============================================================================
# User-defined city-region combinations
# =============================================================================

CITY_REGIONS_TO_RUN = {
    "GSO": ["rural", "industrial", "urban-suburban"],
    "AUS": ["P1R", "P1U", "P2U"],
    "SFO": ["P1R", "P1U", "P2U"],
}

reduced_columns = [
    "total_site_electricity_kw",
    "pf",
]


# =============================================================================
# Base SMART-DS path
# =============================================================================

smart_ds_year = config["smart_ds_years"][0]

smart_ds_load_path = os.path.join(
    config["smart_ds_load_path"],
    str(smart_ds_year),
)


# =============================================================================
# Create reduced measured-building dictionaries
# =============================================================================

for city, regions in CITY_REGIONS_TO_RUN.items():
    for region in regions:

        print(f"\n--- Processing {city} - {region} ---")

        input_data_region_dir = os.path.join(
            smart_ds_load_path,
            city,
            region,
            "buildings",
        )

        input_file_path = os.path.join(
            input_data_region_dir,
            "measured_buildings_cool_heat_dict.joblib",
        )

        output_file_path = os.path.join(
            input_data_region_dir,
            "measured_buildings_total_kw_pf_dict.joblib",
        )

        # ---------------------------------------------------------------------
        # Skip the city-region combination if the reduced file already exists
        # ---------------------------------------------------------------------
        if skip_output_that_already_exists:
            if os.path.exists(output_file_path):
                print(
                    "Reduced dictionary already exists. Skipping:\n"
                    f"{output_file_path}"
                )
                continue

        # ---------------------------------------------------------------------
        # Check that the original dictionary exists
        # ---------------------------------------------------------------------

        if not os.path.exists(input_file_path):
            print(
                "Original dictionary was not found. Skipping:\n"
                f"{input_file_path}"
            )
            continue

        try:
            # -----------------------------------------------------------------
            # Load the original measured-building dictionary
            # -----------------------------------------------------------------

            print(f"Loading:\n{input_file_path}")

            measured_buildings_total_kw_pf_dict = joblib.load(
                input_file_path
            )

            # -----------------------------------------------------------------
            # Retain only total kW and power-factor columns
            #
            # The dictionary is modified in place so that the entire original
            # and reduced dictionaries are not simultaneously stored in RAM.
            # -----------------------------------------------------------------

            print(f"Retaining columns: {reduced_columns}")

            for outer_key, buildings_dict in (
                measured_buildings_total_kw_pf_dict.items()
            ):
                for building_name in list(buildings_dict.keys()):

                    building_df = buildings_dict[building_name]

                    missing_columns = [
                        column
                        for column in reduced_columns
                        if column not in building_df.columns
                    ]

                    if missing_columns:
                        raise KeyError(
                            f"Missing columns {missing_columns} for "
                            f"outer key {outer_key}, building "
                            f"{building_name}."
                        )

                    buildings_dict[building_name] = (
                        building_df.loc[:, reduced_columns]
                        .astype("float32")
                        .copy()
                    )

                    # Release the reference to the full DataFrame.
                    del building_df

            # -----------------------------------------------------------------
            # Save the reduced dictionary
            # -----------------------------------------------------------------

            print(f"Saving:\n{output_file_path}")

            joblib.dump(
                measured_buildings_total_kw_pf_dict,
                output_file_path,
            )

            print(f"Successfully completed {city} - {region}")

        except Exception as error:
            print(
                f"Failed to process {city} - {region}:\n"
                f"{type(error).__name__}: {error}"
            )

        finally:
            # -----------------------------------------------------------------
            # Delete the current city-region dictionary and clear memory before
            # processing the next file.
            # -----------------------------------------------------------------

            if "measured_buildings_total_kw_pf_dict" in locals():
                del measured_buildings_total_kw_pf_dict

            gc.collect()

            print(f"Memory cleared for {city} - {region}")
            
end_time = time.time(); print("Runtime:", (end_time - start_time) / 60, "minutes")